# Land use from LCS Meta-analysis 

THINGS TO DO:
- Land use from m2/FU to m2 --> need to multiply by production, where is the produciton in the spreadsheet?
    - Multiply land use values for arable (animal) products by the total UK production. This should result in a total land area, likely to be different from the total arable (pasture) area in the UK
- Scale all the land use factors associated with arable and animal products so the total arable and pasture land matches the totals in the UK

This notebook is based on the spreadsheet [LCA-metanalysis](https://docs.google.com/spreadsheets/d/11o750uJOFyDJCoJaV7kA5Xz9_qBPsj4NwPPEryt8AZE/edit?gid=1451155914#gid=1451155914). From here you can download the csv file and replace the file name in this notebook.

**First step**
Create dataframe from csv file and assign to each product its category (there are 43 in total) by creating a new column in the dataframe.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/LCA+Meta-Analysis+of+Food+Products+-+Full+Model+v0+(1).csv', skiprows=2, dtype=str)

# detect Category rows
df['xx_num'] = pd.to_numeric(df['xx'], errors='coerce')
is_category_row = df['xx_num'].notna()

# assign Category
df['Category'] = df['Reference'].where(is_category_row)
df['Category'] = df['Category'].ffill()

# drop all rows where 'Product' is NaN
df = df[df['Product'].notna()]

print(len(df['Category'].unique()))

/home/s2561233/anaconda3/envs/agrifoodpy/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


43


**Second step** 
Add another column indicating the functional unit and one with densities (Kg per litre at 20C). Both information from Standardization sheet in LCA meta-analysis.

In [2]:
# the spreadsheet has one value for FU of Bovine meat without distinguish between meat and dairy heard
# added one entry in functional_units to match the number of categories
functional_units = ['1kg', '1kg', '1kg', '1kg', '1kg', 
                    '1kg', '1kg', '1kg', '1kg', '1kg', 
                    '1kg', '1kg', '1kg', '1 litre', '1kg', 
                    '1 litre', '1 litre', '1 litre', '1 litre', '1 litre', 
                    '1kg', '1kg', '1kg', '1kg', '1kg', 
                    '1kg', '1kg', '1kg', '1kg', '1 litre', 
                    '1kg', '1kg', '1kg', '1kg', '1kg', '1kg',
                    '1kg', '1kg', '1kg', '1kg', '1kg', '1kg', '1kg']
liquid_density = {'Olives (Oil)': 0.86,
                  'Palm (Oil)': 0.89,
                  'Rapeseed (Oil)': 0.92, 
                  'Soybeans (Oil)': 0.93,
                  'Soybeans (Soymilk)': 1.01,
                  'Sunflower (Oil)': 0.92}

# each Category has its own functional unit, so I need to add a column with the corresponding functional unit for each category
functional_units_dict = dict(zip(df['Category'].unique(), functional_units))
df['FU'] = df['Category'].map(functional_units_dict)

# add one column with density per each product, if product name not in liquid_density, assign 1 
df['Density'] = df['Category'].map(liquid_density).fillna(1)

**Third step**
We can get rid of most of the columns, just keep product, country, all land use, category and density. We're also interested in UK only, so select for that too.

In [3]:
# keep only relevant columns for land use calculation (Product, Category, Country, Weight and land use types)
df = df[['Product', 'Category', 'Country', 'Weight', 'Land Use (m2*yr)', 'Seed', 'Arable', 'Temp Past', 'Fallow', 'Perm Past', 'Loss', 'Density']]

# create df for European countries (probably not useful)
european_countries = ['Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland', 'France', 'Germany', 'Ireland', 'Italy', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'Romania', 'Russian Federation', 'Spain', 'Sweden', 'Switzerland', 'United Kingdom'] # where did I took these from??
df_europe = df[df['Country'].isin(european_countries)]

# create df for UK only 
df_uk = df[df['Country'] == 'United Kingdom']
df_uk.head()

,Product,Category,Country,Weight,Land Use (m2*yr),Seed,Arable,Temp Past,Fallow,Perm Past,Loss,Density
245,Winter wheat,Wheat/Rye (Bread),United Kingdom,0.1%,1.3,5E-02,1.0,-,0.1,-,0.2,1.0
246,Wheat,Wheat/Rye (Bread),United Kingdom,0.5%,1.2,3E-02,1.0,-,0.0,-,0.2,1.0
247,Winter wheat,Wheat/Rye (Bread),United Kingdom,0.4%,1.4,2E-02,1.2,-,0.0,-,0.2,1.0
248,Winter wheat,Wheat/Rye (Bread),United Kingdom,0.4%,2.0,4E-02,1.7,-,0.0,-,0.3,1.0
249,Winter wheat,Wheat/Rye (Bread),United Kingdom,0.1%,1.5,3E-02,1.1,-,0.1,-,0.2,1.0


**Fourth step** Average over all product types in each category: we want one (weighted average) value of all types of land used for each category
$\Rightarrow$ first weighed average for each product, then average of all product in each category

In [4]:
land_use_types = ['Land Use (m2*yr)', 'Seed', 'Arable', 'Temp Past', 'Fallow', 'Perm Past', 'Loss']

def _to_num(s):
    '''
    Converts s (pd Series) to numeric value replacing '-' with 0 and removing commas for thousands separator. 
    If conversion fails, returns nan.'''
    return pd.to_numeric(
        s.astype(str)
         .str.replace(',', '', regex=False)
         .replace('-', np.nan),
        errors='coerce'
    ).fillna(0)

def create_land_use_dataframe_categories(dataframe, land_use_types=land_use_types):
    """
    Creates a new dataframe with keys: Category and input land use types.
    Each row corresponds to a unique category in the original dataframe with corresponding average value for each land use type
    """
    d = dataframe.copy()
    # convert weights to numbers from str percetnages
    d['Weight_num'] = pd.to_numeric(d['Weight'].astype(str).str.replace('%', '', regex=False), errors='coerce').fillna(0) / 100

    # convert land-use columns to numeric
    for col in land_use_types:
        d[col] = _to_num(d[col])

    # weighted average per category
    rows = []
    for category, group in d.groupby('Category', sort=False):
        w = group['Weight_num'].to_numpy()
        row = {'Category': category}
        for col in land_use_types:
            values = group[col].to_numpy()
            if w.sum() > 0:
                row[col] = np.average(values, weights=w)
            else:
                # when weights are all zeros assign zero
                row[col] = 0
        rows.append(row)

    return pd.DataFrame(rows)

df_uk_cat_avg = create_land_use_dataframe_categories(df_uk)
print(df_uk_cat_avg['Category'].unique())
df_uk_cat_avg.head()

<StringArray>
[           'Wheat/Rye (Bread)',                 'Maize (Meal)',
                'Barley (Beer)',                     'Potatoes',
                   'Sugar Beet',               'Beans & Pulses',
               'Rapeseed (Oil)',                     'Tomatoes',
             'Onions and Leeks',              'Root Vegetables',
 'Cabbages and Other Brassicas',             'Other Vegetables',
                       'Apples',                      'Berries',
      'Bovine Meat (Beef Herd)',     'Bovine Meat (Dairy Herd)',
           'Mutton & Goat Meat',                     'Pig Meat',
                 'Poultry Meat',                         'Milk',
                       'Cheese',                         'Eggs',
                'Fish (farmed)']
Length: 23, dtype: str


,Category,Land Use (m2*yr),Seed,Arable,Temp Past,Fallow,Perm Past,Loss
0,Wheat/Rye (Bread),1.527273,0.087727,1.172727,0.0,0.050000,0.0,0.227273
1,Maize (Meal),1.950000,0.805000,0.700000,0.0,0.200000,0.0,0.250000
2,Barley (Beer),0.359091,0.220227,0.040909,0.0,0.000000,0.0,0.045909
3,Potatoes,0.400000,0.162500,0.100000,0.0,0.000000,0.0,0.080000
4,Sugar Beet,1.711628,0.000091,1.169767,0.0,0.102326,0.0,0.362791


In [5]:
# As a check let's print the total land in UK from this calculations and compare with calculator's
print('Total land from this notebook (m2*yr):', df_uk_cat_avg['Land Use (m2*yr)'].sum())

Total land from this notebook (m2*yr): 293.9473087376871


**Fifth step** Consider and correct for different FU $\rightarrow$ needed for multiply by production       


$\rm land\,use[m^2/kg] \times production\,mass[kg]\,\,\,$  and  $\,\,\,\rm mass[kg] = mass[L] \times density[kg/L]\,\,\,\,$  $\Rightarrow\,\,\,\,$ 
just multiply $\,\,\rm density \times land\,use$

In [6]:
# temporary solution: replace nan with 0 in df_uk_cat_avg
df_uk_cat_avg = df_uk_cat_avg.fillna(0)

# change 'Cabbage and Other Brassicas' to 'Cabbage & Other Brassicas' to match the category name in the calculator
df_uk_cat_avg['Category'] = df_uk_cat_avg['Category'].replace('Cabbage and Other Brassicas', 'Cabbage & Other Brassicas')

**Sixth step**
Match with FAOSTAT items. Note: not all items are produced in UK, only a subselection of items (Categories) is present in the dataframe.

THIS IS DONE BY HAND! Create a disctionary matching FAOSTAT and PN18 items following the [Matching spreadsheet](https://docs.google.com/spreadsheets/d/1M9WcEpb1CC9VkOohmx3rjMk52ZVQWStc1OpUcNIdZ60/edit?gid=398322321#gid=398322321)  (not strictly following as the PN18 items are slightly different, why??)

In [ ]:
faostat_items = {'Wheat and products': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Wheat/Rye (Bread)'].index[0]],
    'Barley and products': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Barley (Beer)'].index[0]],
    'Maize and products': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Maize (Meal)'].index[0]],
        # 'Rye and products', ###### this is already contained in wheat/rye (???)
    # 'Oats', 
    # 'Millet and products', 
    # 'Sorghum and products', 
    # 'Cereals, Other', 
    'Potatoes and products': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Potatoes'].index[0]],
    # 'Cassava and products', 
    # 'Sweet potatoes', 
    'Roots, Other': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Root Vegetables'].index[0]],
    # 'Yams', 
    # 'Sugar cane', 
    'Sugar beet': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Sugar Beet'].index[0]],
    # 'Sugar non-centrifugal', 
    # 'Sugar (Raw Equivalent)', 
    # 'Sweeteners, Other', 
    'Beans': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Beans & Pulses'].index[0]],
    # 'Peas', 
        # 'Pulses, Other and products',  ###### already in beans & pulses (???)
    # 'Nuts and products', 
    # 'Soyabeans', 
    # 'Groundnuts (Shelled Eq)', 
    # 'Sunflower seed', 
    # 'Rape and Mustardseed', 
    # 'Cottonseed', 
    # 'Coconuts - Incl Copra', 
    # 'Sesame seed', 
    # 'Palm kernels', 
    # 'Olives (including preserved)', 
    # 'Oilcrops, Other', 
    # 'Soyabean Oil', 
    # 'Groundnut Oil', 
    # 'Sunflowerseed Oil', 
    'Rape and Mustard Oil': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Rapeseed (Oil)'].index[0]],
    # 'Cottonseed Oil', 
    # 'Palmkernel Oil', 
    # 'Palm Oil', 
    # 'Coconut Oil', 
    # 'Sesameseed Oil', 
    # 'Olive Oil', 
    # 'Ricebran Oil', 
    # 'Maize Germ Oil', 
    # 'Oilcrops Oil, Other', 
    'Tomatoes and products': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Tomatoes'].index[0]],
    'Onions': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Onions and Leeks'].index[0]],
    'Vegetables, Other': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Other Vegetables'].index[0]] +\
        df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Cabbages and Other Brassicas'].index[0]],
    # 'Oranges, Mandarines', 
    # 'Lemons, Limes and products', 
    # 'Grapefruit and products', 
    # 'Citrus, Other', 
    # 'Bananas', 
    # 'Plantains', 
    'Apples and products': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Apples'].index[0]],
    # 'Pineapples and products', 
    # 'Dates', 
    # 'Grapes and products (excl wine)', 
    'Fruits, Other': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Berries'].index[0]],
    # 'Coffee and products', 
    # 'Cocoa Beans and products', 
    # 'Tea (including mate)', 
    # 'Pepper', 
    # 'Pimento', 
    # 'Cloves', 
    # 'Spices, Other', 
    # 'Wine', 
    # 'Beer', # ------> already in barley (from FU (kg) it looks like it counts as cereal/seed => matched with Barley and products instead of here)
    # 'Beverages, Fermented', 
    # 'Beverages, Alcoholic', 
    # 'Alcohol, Non-Food', 
    # 'Infant food', 
    'Bovine Meat': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Bovine Meat (Beef Herd)'].index[0]] +\
        df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Bovine Meat (Dairy Herd)'].index[0]],
    'Mutton & Goat Meat': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Mutton & Goat Meat'].index[0]],
    'Pigmeat': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Pig Meat'].index[0]],
    'Poultry Meat': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Poultry Meat'].index[0]],
    # 'Meat, Other', 
    # 'Offals, Edible', 
    # 'Fats, Animals, Raw', 
    # 'Butter, Ghee', 
    # 'Cream', 
    # 'Honey', 
            # 'Freshwater Fish',   ## too many fishhhh! which one to choose?
            # 'Demersal Fish', 
            # 'Pelagic Fish', 
            # 'Marine Fish, Other', 
            # 'Crustaceans', 
            # 'Cephalopods', 
            # 'Molluscs, Other', 
    # 'Meat, Aquatic Mammals', 
    # 'Aquatic Animals, Others', 
    # 'Aquatic Plants', 
    # 'Fish, Body Oil', 
    # 'Fish, Liver Oil', 
    # 'Rice and products', 
    'Milk - Excluding Butter': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Milk'].index[0]],
    'Eggs': df_uk_cat_avg['Land Use (m2*yr)'][df_uk_cat_avg[df_uk_cat_avg['Category'] == 'Eggs'].index[0]],}

# STILL NEEDS TO BE ASSIGNED: Cheese and Fish (farmed)  !!!!!!!!!!!!!!!!!!!

**Seventh step**
Multiply by production to get land use factor per each FAOSTAT item

In [8]:
fao_plant_production = {'Alcohol, Non-Food': 0.,
    'Apples and products': 6.32E+08,
    'Aquatic Plants': 0.,
    'Bananas': 0.,
    'Barley and products': 8.12E+09,
    'Beans': 0.,
    'Beer': 3.24E+09,
    'Beverages, Alcoholic': 8.85E+08,
    'Beverages, Fermented': 0.,	
    'Cassava and products': 0.,	
    'Cereals, Other': 9.40E+07,
    'Citrus, Other': 1.00E+06,
    'Cloves': 0.,
    'Cocoa Beans and products': 0.,
    'Coconut Oil': 0.,
    'Coconuts - Incl Copra': 0.,
    'Coffee and products': 0.,
    'Cottonseed': 0.,
    'Cottonseed Oil': 0.,
    'Dates': 0.,
    'Fruits, Other': 2.16E+08,
    'Grapefruit and products': 0.,
    'Grapes and products (excl wine)': 1.00E+06,
    'Groundnut Oil': 0.,
    'Groundnuts (Shelled Eq)': 0.,
    'Honey': 9.00E+06,
    'Lemons, Limes and products': 0.,
    'Maize and products': 0.,
    'Maize Germ Oil': 3.20E+07,
    'Millet and products': 0.,
    'Nuts and products': 0.,
    'Oats': 1.03E+09,
    'Oilcrops Oil, Other': 5.20E+07,
    'Oilcrops, Other': 6.30E+07,
    'Olive Oil': 0.,
    'Olives (including preserved)': 0.,
    'Onions': 3.96E+08,
    'Oranges, Mandarines': 0.,
    'Palm kernels': 0.,
    'Palm Oil': 0.,
    'Palmkernel Oil': 0.,
    'Peas': 1.60E+08,
    'Pepper': 0.,
    'Pimento': 0.,
    'Pineapples and products': 0.,
    'Plantains': 0.,
    'Potatoes and products': 5.51E+09,
    'Pulses, Other and products': 6.90E+08,
    'Rape and Mustard Oil': 5.81E+08,
    'Rape and Mustardseed': 1.04E+09,
    'Rice and products': 0.,
    'Ricebran Oil': 0.,
    'Roots, Other': 0.,
    'Rye and products': 7.20E+07,
    'Sesame seed': 0.,
    'Sesameseed Oil': 0.,
    'Sorghum and products': 0.,
    'Soyabean Oil': 1.34E+08,
    'Soyabeans': 0.,
    'Spices, Other': 0.,
    'Sugar (Raw Equivalent)': 9.06E+08,
    'Sugar beet': 5.98E+09,
    'Sugar cane': 0.,
    'Sugar non-centrifugal': 0.,
    'Sunflower seed': 0.,
    'Sunflowerseed Oil': 0.,
    'Sweet potatoes': 0.,
    'Sweeteners, Other': 4.66E+08,
    'Tea (including mate)': 0.,
    'Tomatoes and products': 6.50E+07,
    'Vegetables, Other': 2.16E+09,
    'Wheat and products': 9.66E+09,
    'Wine': 1.00E+06,
    'Yams': 0.,
}

In [9]:
plant_uk_land = []
for item in faostat_items:
    if item in fao_plant_production:
        plant_uk_land[item] = fao_plant_production[item] * faostat_items[item]

TypeError: list indices must be integers or slices, not str

# Old Notebook version (will be deleted)

### Land and production
In order to get the land use factor per each item we need the total land for animal and one for plant product (pastoral and arable land). Therefore to first get the total land used for both in m2, we can just multiply the total land used in m2/FU by the total animal/plant production (of UK).

Columns LH onward in spreadsheet should indicate production (all kind of production, from produced gas to marketable product) -- quite unsure how to interpret all of them, expecially for plant products.
Animal production should be indicated by column PL which contains the ``Marketable final weight (kg liveweight)'' (which I don't think is the only column that has to be used) $\rightarrow$ following cell uses this value to get land used in $\rm m^2$ as: 

$\rm tot\, partoral\, land\, [m^2] =\, land\, [m^2/kg] \times production\, [kg]$

**Question:** How to do this for plant products? Is this even correct at all? I believe that the production (for animal products) is more complicated than just taking values in column PL...

In [ ]:
# select ''M F Wt (kg LW)' in dataset_amimal for United Kingdom only
animal_production_UK = dataset_csv_animal[dataset_csv_animal['Country'] == 'United Kingdom']['M F Wt (kg LW)']
# replace '-' with 0 and convert to float
animal_production_UK = animal_production_UK.replace('-', '0').astype(float)
w_animal = dataset_csv_animal[dataset_csv_animal['Country'].isin(['United Kingdom'])]['Weight'].str.replace('%', '').astype(float) / 100
prod_x_land = past_land_arr_UK * animal_production_UK * w_animal

print('animal produciton', animal_production_UK.sum())
print(f'total pastoral land (sum of all item) = {prod_x_land.sum():.2f} m2  --> first land*prod per each item then summed together')
print(f"(tot past land/kg) * production = {(tot_past_land_UK * animal_production_UK.sum()):.2f} m2  --> sum first (get tot land and tot production) and then multiply the two.")

NameError: name 'dataset_csv_animal' is not defined

The result is different between the two methods because some items have zero in production or land, but if you sum first you don't account for this, _i.e._ if prod=[1, 0, 3, ...] and land=[0, 2, 1, ...] you get 4*3=12 if you first sum then multiply and you get 3 if you do the opposite. 
It makes more sense to me if we first multiply, but if we want to compare with the result obtained with FAOSTAT data we need to sum first.

We do a similar thing for plant items looking at column OH (Yield (kg/l mkt))

In [ ]:
# select 'Yield (kg/l mkt)' in plant for United Kingdom only
plant_production_UK = dataset_csv_plant[dataset_csv_plant['Country'] == 'United Kingdom']['Yield (kg/l mkt)']
# replace '-' with 0 and convert to float
plant_production_UK = plant_production_UK.replace('-', '0').str.replace(',', '').astype(float)
w_plant = dataset_csv_plant[dataset_csv_plant['Country'] == 'United Kingdom']['Weight'].str.replace('%', '').astype(float) / 100
prod_x_land = arable_land_arr_UK * plant_production_UK * w_plant

print('plant produciton', plant_production_UK.sum())
print(f'total pastoral land (sum of all item) = {prod_x_land.sum():.2f} m2  --> first land*prod per each item then summed together')
print(f"(tot past land/kg) * production = {(tot_arable_land_UK * plant_production_UK.sum()):.2f} m2  --> sum first (get tot land and tot production) and then multiply the two.")

Here instead we calculate the same total pastoral land, but using tot animal production from FAOSTAT 

In [ ]:
# values taken from matching spreadsheet (which are taken from agrifoodpy, taken from FAO); values in kg
tot_UK_mass_produciton_animal_FAOSTAT = 21736700000
tot_UK_mass_produciton_plant_FAOSTAT = 42192000000

# land_used * produciton
tot_past_land_UK_FAOSTAT = tot_past_land_UK * tot_UK_mass_produciton_animal_FAOSTAT
tot_arable_land_UK_FAOSTAT = tot_arable_land_UK * tot_UK_mass_produciton_plant_FAOSTAT
print(f'tot pastoral land (land [m2/kg] * production [kg]) for UK: {tot_past_land_UK_FAOSTAT/1e10} *10^10 m2')
print(f'tot arable land (land [m2/kg] * production [kg]) for UK: {tot_arable_land_UK_FAOSTAT/1e10} *10^10 m2')

Tot pastoral and arable land from calculator is or the order of $10^{10}\rm m^2$ -- expected to be different, but this is a bit too much I would say. 

### Land use factor per item

Calculated for each item (UK), returns a dataset.

$\rm land\, use\, factor\, (item)\, = \frac{\rm land\, used\, item}{\rm all\, land\, used\, for\, agri \, or \, past}$

In [ ]:
land_use_factor_per_item = prod_x_land / (tot_past_land_UK_FAOSTAT + tot_arable_land_UK_FAOSTAT)
land_use_factor_per_item

### Scale land factors
Scale all the land use factors associated with arable and animal products so the total arable and pasture land matches the totals in the UK